In [99]:
import pandas as pd
import numpy as np
import re

#orginele dataset inlezen
df = pd.read_csv('Leesmateriaal.csv')
df = df.convert_dtypes()

In [100]:
#Analyse van de kolommen Themas_boek, trefwoorden en bron_url
def normaliseer_waarde(waarde):
    if pd.isna(waarde):
        return ""

    waarde = str(waarde).strip().casefold()

    # Spaties rondom puntkomma's negeren
    waarde = re.sub(r"\s*;\s*", ";", waarde)

    # Meerdere spaties vervangen door één spatie
    waarde = re.sub(r"\s+", " ", waarde)

    return waarde

# Waarden normaliseren
themas = df["Themas_boek"].map(normaliseer_waarde)
trefwoorden = df["trefwoorden"].map(normaliseer_waarde)
bron_url = df["bron_url"].map(normaliseer_waarde)

# Eerst bepalen of Themas_boek een URL bevat
is_url = themas.str.startswith(("http://", "https://"))

# Als Themas_boek een URL is, vergelijk met bron_url.
# Anders vergelijk met trefwoorden.
verwachte_waarde = trefwoorden.copy()
verwachte_waarde.loc[is_url] = bron_url.loc[is_url]

# Controleren welke rijen niet overeenkomen
verschilt = themas != verwachte_waarde

# Alleen afwijkende rijen weergeven
verschillen = df.loc[
    verschilt,
    [
        "Titel",
        "Themas_boek",
        "trefwoorden",
        "bron_url",
    ],
].copy()

# Laat zien waarmee Themas_boek per rij is vergeleken
verschillen["vergeleken_met"] = is_url.loc[verschillen.index].map(
    {
        True: "bron_url",
        False: "trefwoorden",
    }
)

if verschillen.empty:
    print("Alle waarden komen overeen.")
else:
    display(verschillen)

,Titel,Themas_boek,trefwoorden,bron_url,vergeleken_met
188,Zolang het tuin duurt 2,"zelfreflectie,doorzetten",zelfreflectie; doorzetten,<NA>,trefwoorden
201,De kleine kust,"jongeren,vriendschap",jongeren; vriendschap,<NA>,trefwoorden
231,Wat Joris niet wist,"doorzetten,liefde",doorzetten; liefde,<NA>,trefwoorden
242,Sleutel zonder kust,liefde/samenleving,liefde; samenleving,<NA>,trefwoorden
246,De brief van Jelle,geheimen|macht,geheimen; macht,<NA>,trefwoorden
254,Een foto voor Sem,"veerkracht,oorlog",veerkracht; oorlog,<NA>,trefwoorden
270,Het witte huis,reizen/oorlog,reizen; oorlog,<NA>,trefwoorden
282,Grens in de haven,"doorzetten,stalking",doorzetten; stalking,<NA>,trefwoorden


In [101]:
#Analyse van de kolommen genre en genre_kopie

# Waarden uit beide kolommen ophalen en normaliseren
genre = df["genre"].map(normaliseer_waarde)
genre_kopie = df["genre_kopie"].map(normaliseer_waarde)

# Alleen vergelijken als genre_kopie oorspronkelijk geen NaN is
heeft_genre_kopie = df["genre_kopie"].notna()

# Verschillen bepalen
verschilt_genre = (
    heeft_genre_kopie
    & (genre != genre_kopie)
)

# Alleen afwijkende rijen weergeven
genre_verschillen = df.loc[
    verschilt_genre,
    [
        "Titel",
        "genre",
        "genre_kopie",
    ],
].copy()

if genre_verschillen.empty:
    print("Alle ingevulde waarden in genre en genre_kopie komen overeen.")
else:
    display(genre_verschillen)

Alle ingevulde waarden in genre en genre_kopie komen overeen.


In [102]:
#Alle overige kolommen verwijderen die niet relevant zijn voor de analyse
df = df.drop(columns=
             ['Unnamed: 3', 'materiaal_id', 'Themas_boek', 'woordenaantal_omschrijving', 
              'gem_woordlengte', 'gem_zinslengte_woorden', 'aandeel_lange_woorden_pct', 
              'aantal_zinnen', 'interne_batchcode', 'administratieve_prioriteit', 
              'genre_kopie', 'Gemiddelde beoordeling'])

In [103]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", df.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": df.nunique(),
    "Aantal missende waarden": df.isna().sum(),
    "Datatype": df.dtypes,
    "Aantal dubbele waarden": df.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

Aantal rijen en kolommen: (283, 21)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
Titel,277,0,string,6
Auteur,231,7,string,45
Korte omschrijving,282,0,string,1
Type materiaal,10,1,string,272
Niveau,4,6,string,273
genre,3,26,string,254
onderwerp,47,181,string,55
trefwoorden,93,181,string,9
taal,4,0,string,279
aantal_paginas,159,17,Int64,107


In [104]:
df["aantal_paginas"] = df["aantal_paginas"].mask(
    df["aantal_paginas"] <= 0,
    np.nan
)

In [105]:
df["taal"] = (
    df["taal"]
    .str.strip()
    .str.lower()
    .replace({
        "ned": "Nederlands",
        "nl": "Nederlands",
        "nederlands": "Nederlands",
        "dutch": "Nederlands"
    })
)

In [106]:
df["Auteur"] = (
    df["Auteur"]
    .str.replace(r"\s*\(redactie\)", "", regex=True)
    .str.strip()
)

df.iloc[139:143]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,...,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode,doelgroep_extra
139,Vanaf 2 augustus wordt duidelijker of het AI i...,Autoriteit Persoonsgegevens,"AI, privacy, deepfakes",online artikel,3F,informatief,<NA>,<NA>,Nederlands,2,...,https://www.autoriteitpersoonsgegevens.nl/actu...,MBO,Aangevuld (deels gesimuleerd),33.3,verhalen,Leeswereld,<NA>,<NA>,CAT-00140,MBO
140,Transparantie-eisen voor AI: wat betekent dit ...,Autoriteit Persoonsgegevens,"AI-verordening, rechten van burgers",online artikel,3F+,informatief,<NA>,<NA>,Nederlands,6,...,https://www.autoriteitpersoonsgegevens.nl/them...,MBO,Aangevuld (deels gesimuleerd),30.7,verhalen,<NA>,<NA>,online,CAT-00141,MBO
141,AI-verordening,Autoriteit Persoonsgegevens,Europese AI-wetgeving,online artikel,3F+,informatief,<NA>,<NA>,Nederlands,6,...,https://www.autoriteitpersoonsgegevens.nl/them...,MBO,gecontroleerd,49.0,verhalen,Educatief Media,<NA>,online,CAT-00142,MBO
142,"AP: AI-Impactbarometer kleurt rood, actie is n...",Autoriteit Persoonsgegevens,"AI, toezicht, discriminatie, recht",online artikel,3F+,<NA>,<NA>,<NA>,Nederlands,4,...,https://www.autoriteitpersoonsgegevens.nl/actu...,MBO,Aangevuld (deels gesimuleerd),44.7,verhalen,Educatief Media,2021,<NA>,CAT-00143,MBO/NT2


In [107]:
df["interessecluster"] = (
    df["interessecluster"]
    .str.strip()
    .str.lower()
    .str.capitalize()
)
df["interessecluster"].value_counts()

interessecluster
Verhalen        268
Maatschappij     13
Vrije tijd        2
Name: count, dtype: Int64

In [108]:
def merge_doelgroep(row):
    doelgroepen = set()

    for kolom in ["doelgroep", "doelgroep_extra"]:
        waarde = row.get(kolom)

        if pd.notna(waarde):
            for doelgroep in str(waarde).split("/"):
                doelgroep = doelgroep.strip().upper()

                if doelgroep:
                    doelgroepen.add(doelgroep)

    return " / ".join(sorted(doelgroepen))


df["doelgroep"] = df.apply(merge_doelgroep, axis=1)

df = df.drop(columns=["doelgroep_extra"])

In [109]:
df["doelgroep"] = df["doelgroep"].replace("", np.nan)

In [110]:
df["doelgroep"].value_counts()

doelgroep
MBO / NT2    212
MBO           65
NT2            3
Name: count, dtype: int64

In [111]:
df["Type materiaal"].value_counts()

Type materiaal
Boek               140
Boek - thriller     37
online artikel      34
Boek - roman        28
dichtbundel         23
blogpost            15
tijdschrift          2
boek                 1
krant (papier)       1
digitale krant       1
Name: count, dtype: Int64

In [112]:
def kapitaliseer_genres(genre):
    if pd.isna(genre):
        return genre

    return " / ".join(
        deel.strip().capitalize()
        for deel in str(genre).split("/")
    )

def verwerk_type_materiaal(df):
    # Bestaande genres eerst normaliseren
    df["genre"] = df["genre"].apply(kapitaliseer_genres)
    
    masker = (
        df["Type materiaal"]
        .fillna("")
        .str.strip()
        .str.lower()
        .str.startswith("boek -")
    )

    for index in df.index[masker]:
        extra_genre = df.at[index, "Type materiaal"].split("-", 1)[1].strip().capitalize()
        bestaand_genre = df.at[index, "genre"]

        if pd.isna(bestaand_genre) or str(bestaand_genre).strip() == "":
            df.at[index, "genre"] = extra_genre
        elif extra_genre.lower() not in str(bestaand_genre).lower():
            df.at[index, "genre"] = f"{bestaand_genre} / {extra_genre}"

    # Maak bijvoorbeeld "Boek - thriller" gelijk aan "Boek"
    df.loc[masker, "Type materiaal"] = "Boek"

    # Type materiaal normaliseren
    df["Type materiaal"] = (
        df["Type materiaal"]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({
            "digitale krant": "krant",
            "krant (papier)": "krant"
        })
    )

    # Boek met of zonder genre wordt Boek
    df.loc[
        df["Type materiaal"].str.match(r"^boek(?:\s*-.*)?$", na=False),
        "Type materiaal"
    ] = "boek"

    # Alleen de eerste letter hoofdletter maken
    df["Type materiaal"] = df["Type materiaal"].str.capitalize()
        
    return df

df = verwerk_type_materiaal(df)
df["Type materiaal"].value_counts()

Type materiaal
Boek              206
Online artikel     34
Dichtbundel        23
Blogpost           15
Tijdschrift         2
Krant               2
Name: count, dtype: Int64

In [113]:
df["beschikbaar_formaat"] = df["beschikbaar_formaat"].str.capitalize().replace("Digitaal", "Online")

In [114]:
df["gegevensstatus"].value_counts()

gegevensstatus
Aangevuld (deels gesimuleerd)    245
gecontroleerd                     19
In behandeling                    11
Name: count, dtype: Int64

In [115]:
df["gegevensstatus"] = df["gegevensstatus"].replace({
    "Aangevuld (deels gesimuleerd)": "Beschikbaar",
    "gecontroleerd": "Niet beschikbaar",
    }).str.capitalize()

df["gegevensstatus"].value_counts()

gegevensstatus
Beschikbaar         245
Niet beschikbaar     19
In behandeling       11
Name: count, dtype: Int64

In [116]:
from datetime import datetime

huidig_jaar = datetime.now().year

df["publicatiejaar"] = pd.to_numeric(
    df["publicatiejaar"].replace("onbekend", pd.NA),
    errors="coerce"
)

df.loc[df["publicatiejaar"] > huidig_jaar, "publicatiejaar"] = pd.NA

df["publicatiejaar"] = df["publicatiejaar"].astype("Int64")

In [117]:
df["bron_url"] = (
    df["bron_url"]
    .str.replace(r"\s*\[.*$", "", regex=True)
    .str.strip()
)

In [118]:
df["onderwerp"] = df["onderwerp"].replace(
    r"^\[.*\]$",
    np.nan,
    regex=True
)

In [119]:
df["trefwoorden"] = df["trefwoorden"].replace(
    r"^\[.*\]$",
    np.nan,
    regex=True
)

In [120]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", df.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": df.nunique(),
    "Aantal missende waarden": df.isna().sum(),
    "Datatype": df.dtypes,
    "Aantal dubbele waarden": df.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

Aantal rijen en kolommen: (283, 20)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
Titel,277,0,string,6
Auteur,230,7,string,46
Korte omschrijving,282,0,string,1
Type materiaal,6,1,string,276
Niveau,4,6,string,273
genre,7,20,str,256
onderwerp,46,183,string,54
trefwoorden,92,183,string,8
taal,1,0,string,282
aantal_paginas,157,28,Int64,98


In [121]:
df.to_csv("Leesmateriaal_OW.csv", index=False)